# ACD example: DHP (`design="dhp"`) on ImprovedB747Env

This notebook demonstrates **DHP** (Dual Heuristic Programming) from Prokhorov & Wunsch.

- Critic learns \(\lambda = \partial J/\partial R\).
- Actor is updated using model Jacobians \(A,B\) and the critic’s \(\lambda\).

For tracking tasks we use \(R(t)=[x(t),\theta_{ref}(t),q_{ref}(t)]\).



In [ ]:
import sys
from pathlib import Path

HERE = Path.cwd().resolve()
ROOT = HERE if (HERE / "tensoraerospace").exists() else HERE.parents[2]
EX_DIR = ROOT / "example" / "dynamic-programming"
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(EX_DIR))

import numpy as np

from tensoraerospace.agent import ADP
from acd_b747_common import make_env_b747_sine, plot_rollout, rollout

print("repo root:", ROOT)



In [ ]:
# ---- User controls ----
DT = 0.1
N_STEPS = 300
REWARD_MODE = "tracking"

SINE_AMP_DEG = 1.0
SINE_FREQ_HZ = 0.05

DESIGN = "dhp"
DEVICE = "cpu"
GAMMA = 0.99
HIDDEN_SIZE = 64

ACTOR_LR = 5e-5
CRITIC_LR = 1e-4
EXPLORATION_STD = 0.02

# Paper-style stabilization: start from a stabilizing controller
DHP_USE_BASELINE = True
DHP_BASELINE_TYPE = "pid"  # "pid" or "pd"
DHP_BASELINE_KP = -24.6295
DHP_BASELINE_KI = -0.2486
DHP_BASELINE_KD = -7.8179
DHP_PID_MODE = "deg"  # "deg" or "norm"

# Actor learns a residual on top of the baseline
DHP_RESIDUAL_SCALE = 0.25
DHP_ACTOR_DELTA_L2 = 1e-3

# Alternating training cycles (Section III)
DHP_CRITIC_CYCLE_EPISODES = 5
DHP_ACTION_CYCLE_EPISODES = 1

TRAIN_EPISODES = 300
MAX_STEPS_PER_EPISODE = N_STEPS



In [ ]:
def make_env():
    return make_env_b747_sine(
        dt=DT,
        n_steps=N_STEPS,
        reward_mode=REWARD_MODE,
        sine_amp_deg=SINE_AMP_DEG,
        sine_freq_hz=SINE_FREQ_HZ,
    )

# Baseline
base_kind = DHP_BASELINE_TYPE if bool(DHP_USE_BASELINE) else "random"
baseline = rollout(
    make_env(),
    agent=None,
    baseline=base_kind,
    pid_kp=DHP_BASELINE_KP,
    pid_ki=DHP_BASELINE_KI,
    pid_kd=DHP_BASELINE_KD,
    pid_mode=DHP_PID_MODE,
    pd_kp=DHP_BASELINE_KP,
    pd_kd=DHP_BASELINE_KD,
)
plot_rollout(baseline, title=f"Baseline ({base_kind})")

agent = ADP(
    env=make_env(),
    design=DESIGN,
    gamma=GAMMA,
    actor_lr=ACTOR_LR,
    critic_lr=CRITIC_LR,
    hidden_size=HIDDEN_SIZE,
    device=DEVICE,
    exploration_std=EXPLORATION_STD,
    # DHP settings
    dhp_use_baseline=DHP_USE_BASELINE,
    dhp_baseline_type=DHP_BASELINE_TYPE,
    dhp_baseline_kp=DHP_BASELINE_KP,
    dhp_baseline_ki=DHP_BASELINE_KI,
    dhp_baseline_kd=DHP_BASELINE_KD,
    dhp_pid_mode=DHP_PID_MODE,
    dhp_residual_scale=DHP_RESIDUAL_SCALE,
    dhp_actor_delta_l2=DHP_ACTOR_DELTA_L2,
    dhp_critic_cycle_episodes=DHP_CRITIC_CYCLE_EPISODES,
    dhp_action_cycle_episodes=DHP_ACTION_CYCLE_EPISODES,
    log_every_updates=500,
)

print("Training...", "episodes=", TRAIN_EPISODES, "design=", DESIGN)
agent.train(num_episodes=int(TRAIN_EPISODES), max_steps=int(MAX_STEPS_PER_EPISODE))

trained = rollout(make_env(), agent=agent, deterministic=True)
plot_rollout(trained, title=f"Trained ({DESIGN})")

